In [2]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)
tokens.shape

torch.Size([7])

In [3]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings_with_attention_context = u_model(tokens)
sentence_meanings_with_attention_context

tensor([[ 0.0985, -0.0228,  0.0925,  0.0479],
        [ 0.0816, -0.0873,  0.1089,  0.0699],
        [ 0.0106,  0.0727,  0.0217, -0.0068],
        [ 0.0249, -0.0569,  0.0750,  0.0467],
        [ 0.0336, -0.0874,  0.0925,  0.0614],
        [ 0.0495, -0.1005,  0.1028,  0.0690],
        [ 0.0852, -0.1761,  0.1465,  0.1066]], grad_fn=<MmBackward0>)

In [4]:
from transformers import Gemma3ForCausalLM

gemma_model = Gemma3ForCausalLM.from_pretrained("google/gemma-3-1b-it")
u_model, gemma_model

/Users/iremdinc/.pyenv/versions/3.13.3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 1178.17it/s, Materializing param=model.norm.weight]                                


(UstaModel(
   (embedding): Embedding(64, 4)
   (pos_embedding): Embedding(32, 4)
   (self_attention): UstaSelfAttention(
     (q_weights): Linear(in_features=4, out_features=4, bias=False)
     (k_weights): Linear(in_features=4, out_features=4, bias=False)
     (v_weights): Linear(in_features=4, out_features=4, bias=False)
   )
 ),
 Gemma3ForCausalLM(
   (model): Gemma3TextModel(
     (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
     (layers): ModuleList(
       (0-25): 26 x Gemma3DecoderLayer(
         (self_attn): Gemma3Attention(
           (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
           (k_proj): Linear(in_features=1152, out_features=256, bias=False)
           (v_proj): Linear(in_features=1152, out_features=256, bias=False)
           (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
           (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
           (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
         )
     

![Attention Diagram](https://lena-voita.github.io/resources/lectures/seq2seq/transformer/qkv_attention_formula-min.png)

In [5]:
q_weights = torch.nn.Linear(4, 3, bias=False)
k_weights = torch.nn.Linear(4, 3, bias=False)
v_weights = torch.nn.Linear(4, 3, bias=False)

q_of_sentence = q_weights(sentence_meanings_with_attention_context)
k_of_sentence = k_weights(sentence_meanings_with_attention_context)
v_of_sentence = v_weights(sentence_meanings_with_attention_context)
print(q_weights.weight)

q_of_sentence.shape, k_of_sentence.shape, v_of_sentence.shape


Parameter containing:
tensor([[ 0.0104,  0.1436,  0.1804, -0.4028],
        [ 0.2416,  0.0514,  0.3449,  0.3534],
        [-0.3938,  0.4802, -0.4917,  0.2874]], requires_grad=True)


(torch.Size([7, 3]), torch.Size([7, 3]), torch.Size([7, 3]))

In [6]:
k_of_sentence.shape

torch.Size([7, 3])

In [7]:
attention_scores = q_of_sentence @ k_of_sentence.T
attention_weights = torch.softmax(attention_scores / k_of_sentence.shape[-1] ** 0.5, dim=1)

context_vector = attention_weights @ v_of_sentence
context_vector

tensor([[ 0.0218,  0.0824, -0.0569],
        [ 0.0218,  0.0824, -0.0569],
        [ 0.0218,  0.0824, -0.0569],
        [ 0.0218,  0.0824, -0.0569],
        [ 0.0218,  0.0824, -0.0569],
        [ 0.0218,  0.0824, -0.0569],
        [ 0.0218,  0.0824, -0.0568]], grad_fn=<MmBackward0>)

In [8]:
from plot_tokens import plot_tokens

u_sentences = [
  {
    "words": q_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "blue",
  },
  {
    "words": k_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "purple",
  },
  {
    "words": v_of_sentence.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "orange",
  },
  {
    "words": context_vector.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "green",
  },
]

plot_tokens(u_sentences, "Query, Key, Value and Context Vector Space")

Causal Self Attention

In [9]:
attention_weights

tensor([[0.1426, 0.1427, 0.1429, 0.1430, 0.1430, 0.1429, 0.1428],
        [0.1425, 0.1427, 0.1430, 0.1431, 0.1430, 0.1429, 0.1427],
        [0.1429, 0.1429, 0.1427, 0.1428, 0.1429, 0.1429, 0.1430],
        [0.1426, 0.1428, 0.1429, 0.1430, 0.1430, 0.1429, 0.1428],
        [0.1426, 0.1427, 0.1430, 0.1430, 0.1430, 0.1429, 0.1428],
        [0.1425, 0.1427, 0.1430, 0.1431, 0.1430, 0.1429, 0.1427],
        [0.1423, 0.1426, 0.1432, 0.1432, 0.1431, 0.1430, 0.1426]],
       grad_fn=<SoftmaxBackward0>)

![Softmax](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F582a6c51701bb584c1cdd6662cc376b9cadb7160-2048x1152.png&w=3840&q=75)

In [10]:
mask = torch.tril(torch.ones(7, 7))
mask

tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])

In [11]:
attention_weights = torch.randn(7, 7)

masked_attention_weights = attention_weights.masked_fill(mask == 0, -torch.inf)
masked_attention_weights

tensor([[ 1.1753,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-1.1064,  0.1571,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-1.1085, -1.8874,  0.7262,    -inf,    -inf,    -inf,    -inf],
        [-0.0033,  0.7923, -0.3385, -2.1984,    -inf,    -inf,    -inf],
        [-1.3196,  1.6485,  1.2849, -0.8159,  0.1643,    -inf,    -inf],
        [-1.2542, -0.0369,  0.1820, -1.2673, -0.5943, -0.1485,    -inf],
        [-0.6901, -0.2829,  0.0991,  0.4938, -1.0002, -0.6830, -0.8332]])

In [12]:
torch.softmax(masked_attention_weights, dim=1)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2204, 0.7796, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1295, 0.0594, 0.8111, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2474, 0.5481, 0.1769, 0.0275, 0.0000, 0.0000, 0.0000],
        [0.0250, 0.4858, 0.3377, 0.0413, 0.1101, 0.0000, 0.0000],
        [0.0688, 0.2326, 0.2895, 0.0679, 0.1332, 0.2080, 0.0000],
        [0.0945, 0.1420, 0.2081, 0.3088, 0.0693, 0.0952, 0.0819]])

In [13]:
mask = torch.tril(torch.ones(7, 7))
masked_attention_weights = attention_weights.masked_fill(mask == 0, -torch.inf)
masked_attention_weights

softmaxed_attention_weights = torch.softmax(masked_attention_weights, dim=1)
softmaxed_attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2204, 0.7796, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1295, 0.0594, 0.8111, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2474, 0.5481, 0.1769, 0.0275, 0.0000, 0.0000, 0.0000],
        [0.0250, 0.4858, 0.3377, 0.0413, 0.1101, 0.0000, 0.0000],
        [0.0688, 0.2326, 0.2895, 0.0679, 0.1332, 0.2080, 0.0000],
        [0.0945, 0.1420, 0.2081, 0.3088, 0.0693, 0.0952, 0.0819]])

In [17]:
dropout_rate = 0
torch.manual_seed(1)
dropout = torch.nn.Dropout(dropout_rate)
dropout(softmaxed_attention_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2204, 0.7796, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1295, 0.0594, 0.8111, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2474, 0.5481, 0.1769, 0.0275, 0.0000, 0.0000, 0.0000],
        [0.0250, 0.4858, 0.3377, 0.0413, 0.1101, 0.0000, 0.0000],
        [0.0688, 0.2326, 0.2895, 0.0679, 0.1332, 0.2080, 0.0000],
        [0.0945, 0.1420, 0.2081, 0.3088, 0.0693, 0.0952, 0.0819]])